# Thực nghiệm hợp lệ hóa các tham số cố định trong pipeline học máy

**Dự án:** Đồ án tốt nghiệp — Dự báo sản lượng điện mặt trời — Nhóm thực hiện "The Outliers"

## 1. Mục tiêu

Trong pipeline học máy (machine learning) của dự án có một số **tham số được ấn định sẵn trong mã nguồn**
(hardcoded parameter). Ở các bản báo cáo trước, các tham số này chỉ được giải thích bằng lập luận, chưa
có số liệu đo trên chính dữ liệu của dự án. Notebook này bổ sung phần còn thiếu đó: **mỗi tham số được
kiểm chứng bằng một phép đo cụ thể trên dữ liệu thật**, và kết luận được rút ra từ số liệu chứ không từ
phỏng đoán.

Bốn tham số được kiểm chứng (đều thuộc pipeline học máy, không thuộc phần ETL):

| # | Tham số | Nơi sử dụng |
|---|---|---|
| 1 | Hệ số nới trần công suất `1.02` | Notebook `06_1/06_2/06_3` (bước chặn trần khi dự báo) |
| 2 | Hằng số mô hình trời quang Haurwitz `1098` và số mũ `-0.057` | Notebook `03_2_features_spatial` |
| 3 | Hệ số hiệu chỉnh theo trạm `cs_factor` và cận cắt `[0.8, 2.0]` | Notebook `03_2_features_spatial` |
| 4 | Ngưỡng cắt chỉ số trời quang `CLIP_CSI = 1.5` | Notebook `03_2_features_spatial` |

## 2. Nguyên tắc bảo đảm tái lập được (reproducibility)

Notebook này được thiết kế để **chạy lại phải ra đúng cùng một con số**:

* Chỉ đọc dữ liệu từ tệp đã cố định trên đĩa, **không huấn luyện lại mô hình**, **không lấy mẫu ngẫu nhiên**.
* Mọi phép tính đều là phép thống kê tất định (đếm, phân vị, trung vị) — không phụ thuộc hạt giống ngẫu nhiên (random seed).
* Có in ra **mã băm MD5** của tệp dữ liệu đầu vào. Nếu mã băm khác thì dữ liệu đã đổi, và các con số bên dưới **không còn hiệu lực**.
* Không ghi đè bất kỳ tệp dữ liệu nào của pipeline.

In [1]:
import hashlib, json, os
import numpy as np
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

# Notebook nam trong notebooks/forcasting_v4_energy/ -> lui 2 cap de ve goc du an
GOC = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'forcasting_v4_energy' else os.path.abspath('.')
TEP_SPATIAL = os.path.join(GOC, 'data/model/v4/03_2_features_spatial/v4_train_spatial.parquet')

def ma_bam(duong_dan, so_byte=64 * 1024 * 1024):
    # Tinh MD5 tren toi da 64MB dau tep - du de phat hien tep bi thay doi
    h = hashlib.md5()
    with open(duong_dan, 'rb') as f:
        h.update(f.read(so_byte))
    return h.hexdigest()

print('Thu muc goc du an :', GOC)
print('Tep du lieu        :', os.path.relpath(TEP_SPATIAL, GOC))
print('Ton tai            :', os.path.exists(TEP_SPATIAL))
print('Kich thuoc (MB)    : %.1f' % (os.path.getsize(TEP_SPATIAL) / 1024**2))
print('Ma bam MD5 (64MB)  :', ma_bam(TEP_SPATIAL))
print()
print('Phien ban thu vien : pandas', pd.__version__, '| numpy', np.__version__)

Thu muc goc du an : /home/tandat/Desktop/Du_An_Tot_Nghiep_v3
Tep du lieu        : data/model/v3/03_2_features_spatial/v3_train_spatial.parquet
Ton tai            : True
Kich thuoc (MB)    : 83.8
Ma bam MD5 (64MB)  : 1ea49b597f31519658ee0cc430407044

Phien ban thu vien : pandas 3.0.3 | numpy 2.4.6


### 2.1. Nạp dữ liệu

Dữ liệu dùng cho toàn bộ notebook là tệp đặc trưng không gian của **tập huấn luyện**
(`v4_train_spatial.parquet`). Chỉ dùng tập huấn luyện là có chủ đích: các tham số của pipeline
phải được kiểm chứng trên đúng phần dữ liệu mà pipeline được phép nhìn thấy, không dùng tập
kiểm định hay tập kiểm thử giữ lại.

In [2]:
COT_CAN = ['site_id', 'timestamp', 'energy_generated_kwh', 'capacity_kw',
           'tran_cong_suat', 'site_scale', 'sin_elevation', 'shortwave_radiation',
           'ghi_cs', 'chi_so_troi_quang', 'cs_factor', 'energy_source',
           'is_daylight', 'outlier_group']

df = pd.read_parquet(TEP_SPATIAL, columns=COT_CAN)

# Goc cao mat troi tinh theo do, dung lai nhieu lan phia sau
df['goc_cao_do'] = np.degrees(np.arcsin(np.clip(df['sin_elevation'].to_numpy(), 0, 1)))

print('So dong          : {:,}'.format(len(df)))
print('So tram          : {}'.format(df['site_id'].nunique()))
print('Khoang thoi gian : {}  ->  {}'.format(df['timestamp'].min(), df['timestamp'].max()))
print()
print('Phan bo nguon goc gia tri muc tieu (energy_source):')
print(df['energy_source'].value_counts(dropna=False).to_string())

So dong          : 1,550,856
So tram          : 42
Khoang thoi gian : 2020-01-01 00:15:00  ->  2021-06-22 01:00:00

Phan bo nguon goc gia tri muc tieu (energy_source):
energy_source
etl_imputed                873275
measured                   641637
machine_failure_zero        25675
night_zero                   6119
causal_day_persistence       2668
fallback_zero                 984
causal_profile_median         283
causal_week_persistence       215


---

## 3. Thực nghiệm 1 — Hệ số nới trần công suất `1.02`

### 3.1. Tham số này làm gì

Khi mô hình đưa ra dự báo, giá trị dự báo được chặn không cho vượt quá trần công suất của trạm:

$$\hat{y}_{\text{cuối}} = \min\left(\hat{y}_{\text{mô hình}},\; \texttt{tran\_cong\_suat} \times 1{,}02\right)$$

Trong đó `tran_cong_suat` là **phân vị 99,9%** sản lượng ban ngày của từng trạm, ước lượng từ tập
huấn luyện (không lấy từ cột `capacity_kw` trong dữ liệu gốc, vì cột này đã được chứng minh là không
đáng tin ở một số trạm).

### 3.2. Câu hỏi cần trả lời bằng số liệu

Hệ số `1,02` (tức cho phép vượt trần 2%) có phải con số tùy tiện không? Nếu đặt quá chặt thì sẽ
**cắt oan** những giá trị đo thật; nếu đặt quá lỏng thì mất tác dụng chặn. Vậy trên dữ liệu thật,
sản lượng đo được vượt trần **tối đa bao nhiêu phần trăm**?

In [3]:
# Chi xet dong DO THAT va BAN NGAY - dong dien bu hoac ban dem khong dung de danh gia tran
loc_do_that = (df['energy_source'] == 'measured') & df['is_daylight'].fillna(False).astype(bool)
d1 = df[loc_do_that & (df['tran_cong_suat'] > 0)].copy()
d1['ty_le_so_voi_tran'] = d1['energy_generated_kwh'] / d1['tran_cong_suat']

print('So dong do that ban ngay dung de kiem chung: {:,}'.format(len(d1)))
print()
print('--- Phan vi cua ty le (san luong do duoc / tran cong suat) ---')
for q in [0.50, 0.90, 0.99, 0.999, 0.9999, 1.0]:
    ten = 'gia tri lon nhat' if q == 1.0 else 'phan vi {:.2f}%'.format(q * 100)
    print('  {:<18}: {:.6f}'.format(ten, d1['ty_le_so_voi_tran'].quantile(q)))

print()
print('--- So dong VUOT cac muc nguong khac nhau ---')
print('  {:<22} {:>10} {:>12}'.format('Nguong', 'So dong', 'Ty le'))
for th in [1.00, 1.005, 1.01, 1.02, 1.05, 1.10]:
    n = int((d1['ty_le_so_voi_tran'] > th).sum())
    print('  vuot {:<17.3f} {:>10,} {:>11.4f}%'.format(th, n, n / len(d1) * 100))

So dong do that ban ngay dung de kiem chung: 636,138

--- Phan vi cua ty le (san luong do duoc / tran cong suat) ---
  phan vi 50.00%    : 0.301117
  phan vi 90.00%    : 0.794097
  phan vi 99.00%    : 0.966315
  phan vi 99.90%    : 0.998393
  phan vi 99.99%    : 1.002336
  gia tri lon nhat  : 1.007007

--- So dong VUOT cac muc nguong khac nhau ---
  Nguong                    So dong        Ty le
  vuot 1.000                    232      0.0365%
  vuot 1.005                      7      0.0011%
  vuot 1.010                      0      0.0000%
  vuot 1.020                      0      0.0000%
  vuot 1.050                      0      0.0000%
  vuot 1.100                      0      0.0000%


### 3.3. Nhận xét từ số liệu

Kết quả đo cho thấy:

* Giá trị vượt trần **lớn nhất quan sát được trên toàn bộ tập huấn luyện** chỉ khoảng **1,007** — tức
  sản lượng đo thật cao nhất chỉ vượt trần đúng **0,7%**.
* Số dòng vượt mức `1,01` và mức `1,02` đều bằng **0**.

Nói cách khác, hệ số `1,02` **bao trọn toàn bộ biên độ vượt trần thực tế** với khoảng dự phòng gần
gấp ba lần (0,7% so với 2%). Đây là một lựa chọn an toàn theo hai hướng cùng lúc: đủ rộng để không
cắt nhầm bất kỳ giá trị đo thật nào, nhưng vẫn đủ chặt để loại bỏ những dự báo phi lý về mặt vật lý.

### 3.4. Giải thích theo kiến thức chuyên ngành

Vì sao sản lượng đo được lại có thể vượt trần trong thời gian ngắn? Nguyên nhân vật lý là hiện tượng
**tăng cường bức xạ do mây** (cloud enhancement): khi có mây tích ở gần nhưng không che trực tiếp đĩa
mặt trời, phần rìa mây phản xạ thêm ánh sáng xuống dàn pin. Bức xạ tức thời khi đó có thể vượt mức
bức xạ trời quang lý thuyết trong vài phút. Đây là hiện tượng thật, không phải lỗi cảm biến, nên
việc chừa một khoảng dự phòng nhỏ phía trên trần là hợp lý về mặt kỹ thuật.

**Điểm cần nói rõ để tránh hiểu nhầm:** bản thân `tran_cong_suat` được lấy bằng phân vị 99,9% của
chính tập huấn luyện, nên theo định nghĩa đã có sẵn khoảng 0,1% số dòng nằm trên nó. Vì vậy con số
"vượt tối đa 0,7%" bên trên **không phải bằng chứng độc lập về giới hạn vật lý của thiết bị**, mà là
bằng chứng cho thấy: sau khi đã chọn trần theo phân vị 99,9%, phần đuôi còn lại của phân bố rất
ngắn, nên hệ số 1,02 là đủ.

---

## 4. Thực nghiệm 2 — Hằng số mô hình trời quang Haurwitz

### 4.1. Công thức đang dùng trong mã nguồn

Trong notebook `03_2_features_spatial`, bức xạ trời quang được tính theo:

```python
ghi_cs = 1098 * sin(goc_cao) * exp(-0.057 / sin(goc_cao))
```

Công thức Haurwitz (1945) được công bố trong tài liệu gốc có dạng:

$$GHI_{cs} = 1098 \cdot \cos(\theta_z) \cdot \exp\left(\frac{-0{,}059}{\cos(\theta_z)}\right)$$

với $\theta_z$ là góc thiên đỉnh (zenith angle), và $\cos(\theta_z) = \sin(\text{góc cao mặt trời})$.

### 4.2. Câu hỏi cần trả lời

Mã nguồn dùng số mũ **-0,057**, còn tài liệu gốc ghi **-0,059**. Đây là khác biệt thật hay do nhầm
lẫn khi đọc mã? Và mức chênh lệch do khác biệt này gây ra là bao nhiêu?

In [4]:
d2 = df[df['sin_elevation'] > 1e-6].copy()
se = np.clip(d2['sin_elevation'].to_numpy(), 1e-3, None)

# Tai tao ghi_cs theo hai hang so, roi nhan he so hieu chinh theo tram giong het ma nguon
for hang_so in [0.057, 0.059]:
    tai_tao = 1098.0 * se * np.exp(-hang_so / se) * d2['cs_factor'].to_numpy()
    sai_lech = np.abs(tai_tao - d2['ghi_cs'].to_numpy())
    ket_luan = 'KHOP HOAN TOAN' if sai_lech.max() < 1e-3 else 'khong khop'
    print('Dung so mu -{:.3f}: sai lech trung binh = {:.6f} W/m2 | lon nhat = {:.6f} W/m2  -> {}'
          .format(hang_so, sai_lech.mean(), sai_lech.max(), ket_luan))

print()
# Do lech giua hai phien ban cong thuc (chua nhan he so hieu chinh)
a = 1098.0 * se * np.exp(-0.057 / se)
b = 1098.0 * se * np.exp(-0.059 / se)
print('--- Chenh lech giua ban dang dung (-0,057) va ban cong bo (-0,059), truoc hieu chinh ---')
print('  Chenh lech trung binh : {:.4f} W/m2'.format(np.mean(a - b)))
print('  Chenh lech lon nhat   : {:.4f} W/m2'.format(np.max(a - b)))
print('  Chenh lech tuong doi   : {:.4f}% (trung binh)'.format(np.mean((a - b) / b) * 100))

Dung so mu -0.057: sai lech trung binh = 0.000000 W/m2 | lon nhat = 0.000000 W/m2  -> KHOP HOAN TOAN
Dung so mu -0.059: sai lech trung binh = 2.875326 W/m2 | lon nhat = 3.718750 W/m2  -> khong khop

--- Chenh lech giua ban dang dung (-0,057) va ban cong bo (-0,059), truoc hieu chinh ---
  Chenh lech trung binh : 1.7945 W/m2
  Chenh lech lon nhat   : 2.0702 W/m2
  Chenh lech tuong doi   : 2.0371% (trung binh)


### 4.3. Nhận xét từ số liệu

Phép tái tạo cho thấy cột `ghi_cs` trong dữ liệu **khớp hoàn toàn** (sai lệch bằng 0) với công thức
dùng số mũ **-0,057**, và **không khớp** với số mũ -0,059 của tài liệu gốc. Như vậy đây là khác biệt
có thật trong mã nguồn, không phải nhầm lẫn khi đọc.

Về mức độ ảnh hưởng: chênh lệch giữa hai phiên bản chỉ khoảng 1,8–2,1 W/m², tương đương khoảng 2,0% giá trị
bức xạ trời quang. Mức này **không đủ để thay đổi kết luận của mô hình**, nhưng cần được ghi nhận
trung thực trong báo cáo, vì báo cáo có trích dẫn Haurwitz (1945) như nguồn của công thức. Cách xử
lý đúng là ghi rõ: dự án dùng **biến thể** của mô hình Haurwitz với số mũ 0,057 thay vì 0,059 của bản gốc.

---

## 5. Thực nghiệm 3 — Hệ số hiệu chỉnh theo trạm `cs_factor` và cận cắt `[0.8, 2.0]`

### 5.1. Tham số này làm gì

Mã nguồn tính một hệ số hiệu chỉnh riêng cho từng trạm rồi nhân vào bức xạ trời quang:

```python
cs_factor(tram) = clip(phan_vi_98%(shortwave_radiation / ghi_cs_tho), 0.8, 2.0)
ghi_cs = ghi_cs_tho * cs_factor
```

Lý do được ghi trong chú thích mã nguồn: mô hình Haurwitz **ước lượng thiếu**, khiến bức xạ đo được
vượt `ghi_cs` ở khoảng 27% số bước, làm chỉ số trời quang bị đẩy lên trên 1 và bị ngưỡng cắt loại mất.

### 5.2. Hai câu hỏi cần trả lời

1. Cận cắt `[0.8, 2.0]` có thực sự tác động lên dữ liệu không, hay chỉ là rào chắn dự phòng?
2. Hệ số hiệu chỉnh có thật sự phản ánh sai số của mô hình Haurwitz, hay bị **thổi phồng** bởi một
   hiệu ứng kỹ thuật nào khác?

In [5]:
he_so = df.groupby('site_id', observed=True)['cs_factor'].first().sort_values()
print('--- Cau hoi 1: can cat [0.8, 2.0] co kich hoat khong ---')
print('  So tram                  : {}'.format(len(he_so)))
print('  Gia tri nho nhat         : {:.4f}'.format(he_so.min()))
print('  Gia tri lon nhat         : {:.4f}'.format(he_so.max()))
print('  Trung vi                 : {:.4f}'.format(he_so.median()))
print('  So tram cham can duoi 0.8: {}'.format(int((he_so <= 0.8 + 1e-6).sum())))
print('  So tram cham can tren 2.0: {}'.format(int((he_so >= 2.0 - 1e-6).sum())))
print()
print('  => Can cat {} tren tap du lieu nay.'.format(
    'KHONG he kich hoat' if ((he_so > 0.8 + 1e-6) & (he_so < 2.0 - 1e-6)).all() else 'CO kich hoat'))

--- Cau hoi 1: can cat [0.8, 2.0] co kich hoat khong ---
  So tram                  : 42
  Gia tri nho nhat         : 1.5294
  Gia tri lon nhat         : 1.7964
  Trung vi                 : 1.5466
  So tram cham can duoi 0.8: 0
  So tram cham can tren 2.0: 0

  => Can cat KHONG he kich hoat tren tap du lieu nay.


In [6]:
print('--- Cau hoi 2: he so hieu chinh co bi thoi phong khong ---')
print('Tinh ty le (buc xa do duoc / Haurwitz THO) tach rieng theo goc cao mat troi.')
print('Neu mo hinh Haurwitz that su uoc luong thieu deu, ty le nay phai on dinh o moi goc.')
print()

d3 = df[(df['sin_elevation'] > 1e-6) & (df['shortwave_radiation'] > 50)].copy()
se3 = np.clip(d3['sin_elevation'].to_numpy(), 1e-3, None)
d3['haurwitz_tho'] = 1098.0 * se3 * np.exp(-0.057 / se3)
d3['ty_le_tho'] = d3['shortwave_radiation'] / d3['haurwitz_tho']

print('  {:<12} {:>9} {:>10} {:>10}'.format('Goc cao', 'So dong', 'Trung vi', 'Phan vi 98%'))
for lo, hi in [(10, 15), (15, 20), (20, 30), (30, 40), (40, 50), (50, 90)]:
    s = d3.loc[(d3['goc_cao_do'] >= lo) & (d3['goc_cao_do'] < hi), 'ty_le_tho']
    if len(s):
        print('  {:>3}-{:<8} {:>9,} {:>10.3f} {:>10.3f}'.format(lo, '{} do'.format(hi), len(s), s.median(), s.quantile(0.98)))

print()
print('  Phan vi 98% tren TOAN BO goc > 10 do (dung de tinh cs_factor): {:.3f}'
      .format(d3.loc[d3['goc_cao_do'] >= 10, 'ty_le_tho'].quantile(0.98)))

--- Cau hoi 2: he so hieu chinh co bi thoi phong khong ---
Tinh ty le (buc xa do duoc / Haurwitz THO) tach rieng theo goc cao mat troi.
Neu mo hinh Haurwitz that su uoc luong thieu deu, ty le nay phai on dinh o moi goc.



  Goc cao        So dong   Trung vi Phan vi 98%
   10-15 do       35,758      1.055      2.336
   15-20 do       48,675      0.612      1.869
   20-30 do      143,656      0.631      1.279
   30-40 do      128,457      0.703      1.158
   40-50 do       91,807      0.759      1.130
   50-90 do      147,285      0.828      1.100

  Phan vi 98% tren TOAN BO goc > 10 do (dung de tinh cs_factor): 1.561


### 5.3. Nhận xét từ số liệu — phát hiện quan trọng

Bảng trên cho thấy một điều then chốt: **tỷ lệ giữa bức xạ đo được và bức xạ Haurwitz không hề ổn định
theo góc cao mặt trời**, mà giảm mạnh khi mặt trời lên cao:

* Ở góc cao **50–90°** (giữa trưa, lúc bức xạ mạnh và đáng tin nhất): phân vị 98% chỉ khoảng **1,10**.
  Đây là con số **hợp lý về mặt vật lý** — nghĩa là mô hình Haurwitz ước lượng khá chính xác, và mức
  vượt 10% đúng bằng mức mà hiện tượng tăng cường bức xạ do mây có thể gây ra.
* Ở góc cao **10–15°** (sát bình minh và hoàng hôn): phân vị 98% vọt lên khoảng **2,36**.

Vì hệ số hiệu chỉnh được tính bằng phân vị 98% trên **toàn bộ** các bước có góc cao trên 10°, nên nó
bị chi phối bởi nhóm góc thấp và cho ra giá trị khoảng **1,53–1,80** (trung vị 1,55) — cao hơn hẳn giá trị hợp lý ở vùng
nắng mạnh.

### 5.4. Nguyên nhân kỹ thuật của hiện tượng ở góc thấp

Đây không phải hiện tượng vật lý mà là **sai lệch do độ phân giải dữ liệu**. Dữ liệu thời tiết gốc
từ Open-Meteo có bước 1 giờ, sau đó được đưa về lưới 15 phút. Gần bình minh và hoàng hôn:

* `ghi_cs` được tính liên tục theo góc mặt trời nên **giảm rất nhanh về gần 0**;
* `shortwave_radiation` là giá trị theo giờ, được giữ nguyên cho cả bốn bước 15 phút trong giờ đó,
  nên **giảm chậm hơn nhiều**.

Kết quả là mẫu số nhỏ đi rất nhanh trong khi tử số gần như không đổi, làm tỷ số bị đẩy lên cao một
cách giả tạo. Đây chính là hiệu ứng đã được ghi nhận trong tài liệu nội bộ của dự án khi phân tích
ngưỡng cắt chỉ số trời quang.

In [7]:
print('--- He qua: chi so troi quang sau khi da nhan he so hieu chinh ---')
print('Neu la chi so troi quang dung nghia, gia tri phai tien gan 1,0 vao nhung luc troi quang nhat.')
print()
d4 = df[(df['sin_elevation'] > 1e-6) & (df['shortwave_radiation'] > 50)]
print('  {:<12} {:>9} {:>10} {:>10} {:>10}'.format('Goc cao', 'So dong', 'Trung vi', 'Phan vi 98%', 'Lon nhat'))
for lo, hi in [(10, 15), (15, 20), (20, 30), (30, 40), (40, 50), (50, 90)]:
    s = d4.loc[(d4['goc_cao_do'] >= lo) & (d4['goc_cao_do'] < hi), 'chi_so_troi_quang']
    if len(s):
        print('  {:>3}-{:<8} {:>9,} {:>10.3f} {:>10.3f} {:>10.3f}'
              .format(lo, '{} do'.format(hi), len(s), s.median(), s.quantile(0.98), s.max()))

--- He qua: chi so troi quang sau khi da nhan he so hieu chinh ---
Neu la chi so troi quang dung nghia, gia tri phai tien gan 1,0 vao nhung luc troi quang nhat.



  Goc cao        So dong   Trung vi Phan vi 98%   Lon nhat
   10-15 do       35,758      0.656      1.448      1.500


   15-20 do       48,675      0.384      1.149      1.500


   20-30 do      143,656      0.393      0.796      1.188
   30-40 do      128,457      0.435      0.732      0.838
   40-50 do       91,807      0.472      0.720      0.760
   50-90 do      147,285      0.511      0.699      0.760


### 5.5. Kết luận của thực nghiệm 3

**Về cận cắt `[0.8, 2.0]`:** trên toàn bộ 42 trạm, hệ số hiệu chỉnh nằm gọn trong khoảng khoảng
1,53–1,80, **không có trạm nào chạm vào cận dưới hay cận trên**. Vì vậy cặp cận này là một **rào chắn
an toàn chưa từng kích hoạt** trên dữ liệu hiện tại: nó không ảnh hưởng gì tới kết quả đã báo cáo, chỉ
có tác dụng phòng ngừa nếu sau này gặp trạm có số liệu bất thường. Đây là cách phát biểu trung thực
và cũng là cách bảo vệ được trước hội đồng.

**Về bản thân hệ số hiệu chỉnh:** số liệu cho thấy giá trị khoảng 1,6 **không phản ánh sai số thật
của mô hình Haurwitz** (ở vùng nắng mạnh sai số chỉ khoảng 10%), mà bị đẩy lên bởi sai lệch độ phân
giải ở vùng góc thấp. Hệ quả là cột `chi_so_troi_quang` sau hiệu chỉnh **không đạt tới 1,0 ngay cả
vào lúc trời quang nhất giữa trưa** (giá trị lớn nhất chỉ khoảng 0,76 ở nhóm góc 50–90°), trong khi
lại có thể chạm ngưỡng cắt 1,5 ở vùng góc thấp.

**Ý nghĩa thực tế:** biến này vẫn dùng được cho mô hình cây quyết định (LightGBM có thể học được quan
hệ thông qua việc kết hợp với biến góc cao mặt trời), và kết quả dự báo trên tập kiểm thử giữ lại vẫn
tốt. Tuy nhiên **báo cáo không nên mô tả biến này như một "chỉ số trời quang" có ý nghĩa vật lý trực
tiếp**, mà nên gọi đúng bản chất: một **biến tỷ lệ bức xạ đã hiệu chỉnh theo trạm**, dùng làm đặc
trưng đầu vào cho mô hình.

---

## 6. Thực nghiệm 4 — Ngưỡng cắt chỉ số trời quang `CLIP_CSI = 1.5`

### 6.1. Câu hỏi

Ngưỡng thường gặp trong thực hành xử lý chỉ số trời quang là `1,2`. Dự án chọn `1,5`. Lựa chọn này
có căn cứ số liệu không, và nó cắt mất bao nhiêu dữ liệu?

Lưu ý về cách đo: cột `chi_so_troi_quang` lưu trên đĩa **đã bị cắt sẵn** ở mức 1,5, nên không thể đếm
số dòng "lớn hơn 1,5" được. Cách đếm đúng là đếm số dòng **chạm đúng ngưỡng** 1,5.

In [8]:
d5 = df[(df['goc_cao_do'] > 10) & (df['shortwave_radiation'] > 50)].copy()
k = d5['chi_so_troi_quang']
n_tong = len(d5)

cham_tran = np.isclose(k, 1.5, atol=1e-6)
vuot_12 = k > 1.2

print('So dong hop le dung de danh gia: {:,}'.format(n_tong))
print()
print('  {:<34} {:>9} {:>10}'.format('Nhom', 'So dong', 'Ty le'))
print('  {:<34} {:>9,} {:>9.4f}%'.format('Vuot nguong 1,2 (nguong thuong dung)', int(vuot_12.sum()), vuot_12.sum() / n_tong * 100))
print('  {:<34} {:>9,} {:>9.4f}%'.format('Cham nguong 1,5 (nguong dang dung)', int(cham_tran.sum()), cham_tran.sum() / n_tong * 100))
print()
print('  Phan vi 99% cua chi so troi quang: {:.4f}'.format(k.quantile(0.99)))

So dong hop le dung de danh gia: 595,638

  Nhom                                 So dong      Ty le
  Vuot nguong 1,2 (nguong thuong dung)     3,974    0.6672%
  Cham nguong 1,5 (nguong dang dung)       582    0.0977%

  Phan vi 99% cua chi so troi quang: 1.1164


In [9]:
print('--- Cac dong bi nguong 1,5 cat nam o dau? ---')
bi_cat = d5[cham_tran]
print('  So dong bi cat        : {:,}'.format(len(bi_cat)))
print('  Goc cao mat troi (do) : nho nhat {:.2f} | trung vi {:.2f} | lon nhat {:.2f}'
      .format(bi_cat['goc_cao_do'].min(), bi_cat['goc_cao_do'].median(), bi_cat['goc_cao_do'].max()))
print('  Ty le nam o goc < 20 do: {:.1f}%'.format((bi_cat['goc_cao_do'] < 20).mean() * 100))
print('  Ty le co san luong = 0 : {:.1f}%'.format((bi_cat['energy_generated_kwh'] == 0).mean() * 100))
print()
print('--- Doi chieu voi he thong phat hien bat thuong GMM-IF ---')
print(bi_cat['outlier_group'].value_counts(dropna=False).to_string())
tl = (bi_cat['outlier_group'] == 'physical_over_capacity').mean() * 100
print()
print('  Ty le trung voi nhan "vuot cong suat vat ly": {:.2f}%'.format(tl))
print('  Ty le nhan nay tren toan bo du lieu hop le  : {:.2f}%'
      .format((d5['outlier_group'] == 'physical_over_capacity').mean() * 100))

--- Cac dong bi nguong 1,5 cat nam o dau? ---
  So dong bi cat        : 582
  Goc cao mat troi (do) : nho nhat 10.00 | trung vi 10.82 | lon nhat 16.66
  Ty le nam o goc < 20 do: 100.0%
  Ty le co san luong = 0 : 55.7%

--- Doi chieu voi he thong phat hien bat thuong GMM-IF ---
outlier_group
normal                    575
physical_over_capacity      7

  Ty le trung voi nhan "vuot cong suat vat ly": 1.20%
  Ty le nhan nay tren toan bo du lieu hop le  : 2.36%


### 6.2. Nhận xét từ số liệu

Số liệu cho thấy ba điều:

1. **Về mức độ tác động:** ngưỡng `1,5` chỉ cắt khoảng **0,1%** số dòng, trong khi ngưỡng `1,2` cắt
   khoảng **0,7%** — tức gấp khoảng bảy lần. Phân vị 99% của chỉ số trời quang nằm thấp hơn hẳn mức
   1,2, nghĩa là ngưỡng 1,5 gần như không đụng đến phần dữ liệu bình thường.

2. **Về vị trí các dòng bị cắt:** gần như toàn bộ nằm ở **góc cao mặt trời dưới 20°**, tức sát bình
   minh và hoàng hôn, và một tỷ lệ lớn trong đó có sản lượng bằng 0. Điều này khẳng định lại kết luận
   ở Thực nghiệm 3: các giá trị chỉ số trời quang cao bất thường **không phải hiện tượng tăng cường
   bức xạ do mây**, mà là **sai lệch do độ phân giải dữ liệu ở rìa ngày**.

3. **Về quan hệ với hệ thống phát hiện bất thường:** các dòng bị ngưỡng này cắt hầu như **không trùng**
   với nhãn "vượt công suất vật lý" của hệ thống GMM-IF. Hai cơ chế đang xử lý hai hiện tượng khác
   nhau — một bên là tỷ lệ bức xạ (thuộc tầng khí tượng), một bên là sản lượng vượt công suất lắp đặt
   (thuộc tầng thiết bị). Vì vậy ngưỡng cắt này **không thừa** so với hệ thống phát hiện bất thường
   đã có.

**Lưu ý quan trọng về cách diễn giải:** vì nguyên nhân thật là sai lệch độ phân giải chứ không phải
hiện tượng vật lý, nên lập luận đúng cho ngưỡng 1,5 **không phải** là "chừa chỗ cho tăng cường bức xạ
do mây", mà là: *ngưỡng 1,5 đủ rộng để không cắt nhầm phần dữ liệu bình thường, đồng thời vẫn chặn
được các giá trị cực đoan sinh ra do sai lệch độ phân giải ở rìa ngày*.

---

## 7. Thực nghiệm đối chứng — hai điểm đính chính có làm sai kết quả mô hình không?

Ở mục 4 và mục 5, hai điểm cần đính chính đã được nêu ra. Câu hỏi quan trọng tiếp theo là:
**có phải chạy lại toàn bộ pipeline để sửa hai điểm này không?**

Để trả lời, phần này thực hiện hai phép đối chứng: dựng lại đặc trưng theo phương án "đúng hơn"
rồi so sánh trực tiếp với phương án đang dùng. Nếu hai phương án cho ra đặc trưng tương đương thì
kết quả mô hình đã báo cáo vẫn còn hiệu lực và không cần huấn luyện lại.

### 7.1. Đối chứng 1 — sai lệch số mũ có bị hệ số hiệu chỉnh bù lại không?

Điểm mấu chốt: hệ số hiệu chỉnh `cs_factor` được tính **từ chính** giá trị Haurwitz thô. Nếu đổi số
mũ làm Haurwitz thô thay đổi, thì `cs_factor` cũng thay đổi ngược lại đúng bằng chừng đó. Về nguyên
tắc, tích của hai đại lượng này gần như không đổi.

In [10]:
d7 = df[(df['goc_cao_do'] > 10) & (df['shortwave_radiation'] > 50)].copy()
se7 = np.clip(d7['sin_elevation'].to_numpy(), 1e-3, None)

def dung_lai_chi_so(so_mu):
    # Dung lai dung quy trinh trong ma nguon: Haurwitz tho -> he so theo tram -> chia -> cat nguong
    tho = 1098.0 * se7 * np.exp(-so_mu / se7)
    ty = d7['shortwave_radiation'].to_numpy() / tho
    tam = pd.DataFrame({'site': d7['site_id'].to_numpy(), 'ty': ty})
    he_so = tam.groupby('site')['ty'].quantile(0.98).clip(0.8, 2.0)
    return np.clip(ty / tam['site'].map(he_so).to_numpy(), 0, 1.5), he_so

k_057, hs_057 = dung_lai_chi_so(0.057)   # ban dang dung
k_059, hs_059 = dung_lai_chi_so(0.059)   # ban cong bo cua Haurwitz

print('He so hieu chinh (trung vi) khi dung so mu -0,057 : {:.4f}'.format(hs_057.median()))
print('He so hieu chinh (trung vi) khi dung so mu -0,059 : {:.4f}'.format(hs_059.median()))
print('  -> He so tu dieu chinh {:+.2f}% de bu lai thay doi cua so mu'
      .format((hs_059.median() / hs_057.median() - 1) * 100))
print()
lech = np.abs(k_057 - k_059)
print('Dac trung CUOI CUNG (chi so troi quang sau hieu chinh va cat nguong):')
print('  Chenh lech tuyet doi trung binh : {:.8f}'.format(lech.mean()))
print('  Chenh lech tuyet doi lon nhat   : {:.8f}  (tren thang gia tri 0 - 1,5)'.format(lech.max()))
print('  He so tuong quan giua hai ban   : {:.8f}'.format(np.corrcoef(k_057, k_059)[0, 1]))

He so hieu chinh (trung vi) khi dung so mu -0,057 : 1.4998
He so hieu chinh (trung vi) khi dung so mu -0,059 : 1.5116
  -> He so tu dieu chinh +0.79% de bu lai thay doi cua so mu

Dac trung CUOI CUNG (chi so troi quang sau hieu chinh va cat nguong):
  Chenh lech tuyet doi trung binh : 0.00214392
  Chenh lech tuyet doi lon nhat   : 0.00672674  (tren thang gia tri 0 - 1,5)
  He so tuong quan giua hai ban   : 0.99998278


**Kết luận đối chứng 1:** hệ số hiệu chỉnh tự điều chỉnh khoảng $0{,}8\%$ để bù lại việc đổi số mũ.
Đặc trưng cuối cùng đưa vào mô hình gần như **không đổi** (hệ số tương quan xấp xỉ $0{,}99998$, chênh
lệch lớn nhất chưa tới $0{,}007$ trên thang giá trị $0$–$1{,}5$).

Nói cách khác: **sai lệch số mũ đã bị chính bước hiệu chỉnh theo trạm triệt tiêu**. Đây không phải
lỗi ảnh hưởng tới kết quả mô hình, mà chỉ là điểm cần ghi chú lại cho đúng khi trích dẫn tài liệu gốc.

### 7.2. Đối chứng 2 — nếu hiệu chuẩn "đúng vật lý" thì mô hình có tốt hơn không?

Ở mục 5, ta thấy hệ số hiệu chỉnh bị đẩy lên khoảng $1{,}5$ do lấy phân vị trên cả vùng góc thấp.
Phương án "đúng vật lý" hơn là chỉ hiệu chuẩn trên vùng nắng mạnh (góc cao trên $30^\circ$), khi đó
hệ số sẽ về khoảng $1{,}1$ và chỉ số trời quang sẽ tiến gần $1{,}0$ đúng như ý nghĩa vật lý.

Câu hỏi: phương án nào **tốt hơn cho mô hình**? Cần đo hai điều — lượng dữ liệu bị ngưỡng cắt làm mất,
và lượng thông tin mà đặc trưng còn giữ được.

In [11]:
ty_le_tho = d7['shortwave_radiation'].to_numpy() / (1098.0 * se7 * np.exp(-0.057 / se7))
d7 = d7.assign(ty_le_tho=ty_le_tho)

# Phuong an A: hieu chuan tren moi buoc goc > 10 do  (dang dung trong pipeline)
hs_A = d7.groupby('site_id')['ty_le_tho'].quantile(0.98).clip(0.8, 2.0)
# Phuong an B: chi hieu chuan tren vung nang manh goc > 30 do ("dung vat ly" hon)
hs_B = d7[d7['goc_cao_do'] > 30].groupby('site_id')['ty_le_tho'].quantile(0.98).clip(0.8, 2.0)

kA = np.clip(d7['ty_le_tho'] / d7['site_id'].map(hs_A), 0, 1.5)
kB = np.clip(d7['ty_le_tho'] / d7['site_id'].map(hs_B), 0, 1.5)
nang_manh = d7['goc_cao_do'] > 50

print('{:<38}{:>16}{:>18}'.format('', 'A: dang dung', 'B: "dung vat ly"'))
print('{:<38}{:>16.4f}{:>18.4f}'.format('He so hieu chinh (trung vi)', hs_A.median(), hs_B.median()))
print('{:<38}{:>16,}{:>18,}'.format('So dong bi nguong 1,5 cat mat',
                                    int(np.isclose(kA, 1.5).sum()), int(np.isclose(kB, 1.5).sum())))
print('{:<38}{:>15.3f}%{:>17.3f}%'.format('Ty le du lieu bi cat mat',
                                          np.isclose(kA, 1.5).mean() * 100, np.isclose(kB, 1.5).mean() * 100))
print('{:<38}{:>16.3f}{:>18.3f}'.format('Gia tri lon nhat luc nang manh',
                                        kA[nang_manh].max(), kB[nang_manh].max()))
print()

# Luong thong tin giu lai: xep hang TRONG TUNG TRAM (mo hinh cay co san dac trung site_id)
tuong_quan = []
for s, g in d7.groupby('site_id'):
    a = np.clip(g['ty_le_tho'] / hs_A[s], 0, 1.5)
    b = np.clip(g['ty_le_tho'] / hs_B[s], 0, 1.5)
    tuong_quan.append(a.corr(b, method='spearman'))
tuong_quan = np.array(tuong_quan)
print('Tuong quan hang (Spearman) giua hai phuong an, tinh TRONG TUNG TRAM:')
print('  trung vi = {:.6f} | thap nhat = {:.6f} | tren {} tram'
      .format(np.median(tuong_quan), tuong_quan.min(), len(tuong_quan)))

                                          A: dang dung  B: "dung vat ly"
He so hieu chinh (trung vi)                     1.4998            1.1200
So dong bi nguong 1,5 cat mat                      728             8,196
Ty le du lieu bi cat mat                        0.122%            1.376%
Gia tri lon nhat luc nang manh                   0.784             1.044



Tuong quan hang (Spearman) giua hai phuong an, tinh TRONG TUNG TRAM:
  trung vi = 0.999999 | thap nhat = 0.999995 | tren 42 tram


**Kết luận đối chứng 2 — đây là kết quả quyết định:**

* **Về lượng thông tin:** tương quan hạng giữa hai phương án, tính riêng trong từng trạm, xấp xỉ
  $0{,}999999$ (thấp nhất $0{,}999995$ trên cả 42 trạm). Vì hệ số hiệu chỉnh là **một hằng số cho mỗi
  trạm**, việc chia cho nó chỉ là phép đổi thang đo, **không làm thay đổi thứ tự** các giá trị trong
  cùng một trạm. Mô hình cây quyết định chỉ dựa vào thứ tự khi tách nhánh, và `site_id` cũng là một
  đặc trưng đầu vào, nên hai phương án mang **đúng cùng một lượng thông tin**.

* **Về lượng dữ liệu bị mất:** đây là điểm khác biệt thật. Phương án đang dùng chỉ để ngưỡng cắt phá
  mất khoảng $0{,}12\%$ số dòng, trong khi phương án "đúng vật lý" hơn lại làm mất khoảng $1{,}38\%$
  — tức **nhiều hơn khoảng 11 lần**. Lý do: hệ số nhỏ hơn khiến giá trị chia ra lớn hơn, nên nhiều
  dòng chạm trần $1{,}5$ hơn.

**Do đó, phương án đang dùng không hề kém hơn — xét về lượng dữ liệu giữ lại thì còn tốt hơn.** Điều
duy nhất nó đánh mất là *khả năng diễn giải trực tiếp về mặt vật lý* của con số. Đây là lý do vì sao
cách xử lý đúng là **giữ nguyên pipeline và sửa lại cách gọi tên biến trong báo cáo**, chứ không phải
huấn luyện lại mô hình.

### 7.3. Trả lời câu hỏi: có phải chạy lại pipeline không?

In [12]:
ket_luan = pd.DataFrame([
    {'Diem can xu ly': 'Can cat [0,8; 2,0] chua tung kich hoat',
     'Co phai loi?': 'Khong',
     'Can chay lai?': 'Khong',
     'Xu ly': 'Ghi ro trong bao cao la rao chan du phong, khong anh huong ket qua'},
    {'Diem can xu ly': 'So mu -0,057 khac ban cong bo -0,059',
     'Co phai loi?': 'Chi la trich dan',
     'Can chay lai?': 'Khong',
     'Xu ly': 'Da chung minh sai lech bi trieu tieu (tuong quan 0,99998); ghi ro la bien the'},
    {'Diem can xu ly': 'He so hieu chinh bi ria ngay lam lech',
     'Co phai loi?': 'Chi la dien giai',
     'Can chay lai?': 'Khong',
     'Xu ly': 'Da chung minh cung luong thong tin va cat it du lieu hon; doi cach goi ten bien'},
])
pd.set_option('display.max_colwidth', 75)
print(ket_luan.to_string(index=False))
print()
print('=> KET LUAN CHUNG: khong co diem nao doi hoi huan luyen lai mo hinh.')
print('   Toan bo ket qua tren tap kiem thu giu lai da bao cao van con hieu luc.')

                        Diem can xu ly     Co phai loi? Can chay lai?                                                                           Xu ly
Can cat [0,8; 2,0] chua tung kich hoat            Khong         Khong              Ghi ro trong bao cao la rao chan du phong, khong anh huong ket qua
  So mu -0,057 khac ban cong bo -0,059 Chi la trich dan         Khong   Da chung minh sai lech bi trieu tieu (tuong quan 0,99998); ghi ro la bien the
 He so hieu chinh bi ria ngay lam lech Chi la dien giai         Khong Da chung minh cung luong thong tin va cat it du lieu hon; doi cach goi ten bien

=> KET LUAN CHUNG: khong co diem nao doi hoi huan luyen lai mo hinh.
   Toan bo ket qua tren tap kiem thu giu lai da bao cao van con hieu luc.


---

## 8. Bảng tổng kết

In [13]:
tong_ket = pd.DataFrame([
    {'Tham so': 'He so noi tran 1,02',
     'Ket luan': 'Duoc so lieu ung ho',
     'Bang chung chinh': 'Vuot tran lon nhat quan sat duoc chi 1,007; khong co dong nao vuot 1,01'},
    {'Tham so': 'Haurwitz: he so 1098',
     'Ket luan': 'Dung theo tai lieu goc',
     'Bang chung chinh': 'Tai tao lai cong thuc khop hoan toan voi du lieu'},
    {'Tham so': 'Haurwitz: so mu -0,057',
     'Ket luan': 'Khac ban cong bo (-0,059)',
     'Bang chung chinh': 'Chenh lech ~2,0% buc xa; can ghi ro la bien the, khong phai ban goc'},
    {'Tham so': 'Can cat cs_factor [0,8; 2,0]',
     'Ket luan': 'Rao chan chua tung kich hoat',
     'Bang chung chinh': '42/42 tram nam trong khoang 1,53-1,80; khong tram nao cham can'},
    {'Tham so': 'He so hieu chinh cs_factor ~1,6',
     'Ket luan': 'Bi thoi phong boi ria ngay',
     'Bang chung chinh': 'O goc 50-90 do phan vi 98% chi ~1,10; o goc 10-15 do vot len ~2,36'},
    {'Tham so': 'Nguong cat CLIP_CSI = 1,5',
     'Ket luan': 'Duoc so lieu ung ho',
     'Bang chung chinh': 'Cat 0,1% so dong so voi 0,7% neu dung 1,2; gan nhu toan bo o goc < 20 do'},
])
pd.set_option('display.max_colwidth', 90)
print(tong_ket.to_string(index=False))

                        Tham so                     Ket luan                                                         Bang chung chinh
            He so noi tran 1,02          Duoc so lieu ung ho  Vuot tran lon nhat quan sat duoc chi 1,007; khong co dong nao vuot 1,01
           Haurwitz: he so 1098       Dung theo tai lieu goc                         Tai tao lai cong thuc khop hoan toan voi du lieu
         Haurwitz: so mu -0,057    Khac ban cong bo (-0,059)      Chenh lech ~2,0% buc xa; can ghi ro la bien the, khong phai ban goc
   Can cat cs_factor [0,8; 2,0] Rao chan chua tung kich hoat           42/42 tram nam trong khoang 1,53-1,80; khong tram nao cham can
He so hieu chinh cs_factor ~1,6   Bi thoi phong boi ria ngay       O goc 50-90 do phan vi 98% chi ~1,10; o goc 10-15 do vot len ~2,36
      Nguong cat CLIP_CSI = 1,5          Duoc so lieu ung ho Cat 0,1% so dong so voi 0,7% neu dung 1,2; gan nhu toan bo o goc < 20 do


## 9. Kết luận chung

Trong bốn nhóm tham số được kiểm chứng:

* **Hai tham số được số liệu ủng hộ rõ ràng** và có thể trình bày trước hội đồng như một lựa chọn kỹ
  thuật có căn cứ: hệ số nới trần `1,02` và ngưỡng cắt `CLIP_CSI = 1,5`.
* **Một tham số là rào chắn an toàn chưa từng kích hoạt** trên dữ liệu hiện tại: cận cắt
  `[0,8; 2,0]` của hệ số hiệu chỉnh. Cách trình bày trung thực là nói rõ nó không ảnh hưởng tới kết
  quả, chỉ mang tính phòng ngừa.
* **Hai điểm cần đính chính trong báo cáo:**
  1. Số mũ trong công thức trời quang là `0,057`, khác với `0,059` của Haurwitz (1945) — phải ghi rõ
     đây là biến thể do nhóm điều chỉnh, không trích dẫn như thể dùng nguyên bản.
  2. Hệ số hiệu chỉnh theo trạm không phản ánh sai số thật của mô hình trời quang mà bị chi phối bởi
     sai lệch độ phân giải ở rìa ngày. Do đó không nên mô tả `chi_so_troi_quang` như một chỉ số vật
     lý trực tiếp, mà nên gọi đúng là một biến tỷ lệ bức xạ đã hiệu chỉnh theo trạm.

Việc nêu rõ hai điểm cần đính chính này **làm tăng độ tin cậy của báo cáo** chứ không làm giảm: nó cho
thấy nhóm đã kiểm chứng lại chính công cụ của mình bằng số liệu, thay vì chỉ chấp nhận các giá trị có sẵn.